# Conditional mechanism-recovery failure by mutation complexity

For each mutation count, $\mathrm{SA}(\mathcal{P})$ and strict task-level $\mathrm{SA}(\mathcal{M})$ use the fixed Core/Full denominators. The third curve is computed from paired task outcomes as $1-\Pr(\mathcal{M}=1\mid\mathcal{P}=1)$. When no task has $\mathcal{P}=1$, the conditional quantity is undefined and retained as `NaN`; the plot shows no data marker there and uses a dashed guide from the last defined point to 100%.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import yaml
from yumeow_plot import get_fig

REPO = Path.cwd()
if not (REPO / 'tasks' / 'task_split.yaml').is_file():
    REPO = REPO.parent
RESULTS = REPO / 'results'
OUTPUT_DIR = REPO / 'logs' / 'analysis' / '260926分析条件概率'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPLEXITIES = np.arange(6)

split = yaml.safe_load((REPO / 'tasks' / 'task_split.yaml').read_text())
CORE_IDENTITIES = {
    (group['domain'], task_name)
    for group in split['core_set']
    for task_name in group['task_names']
}
FULL_GROUPS = {(group['domain'], group['task_group'])
               for group in split['full_set']}


In [ ]:
def variant_complexity(task_name: str) -> int:
    suffix = task_name.rsplit(' - ', 1)[-1]
    return 0 if suffix == 'Original' else len(
        suffix.removeprefix('Variant ').split('-'))


TASK_METADATA = {}
for task_path in (REPO / 'tasks').glob('*/*/*.yaml'):
    domain, task_group = task_path.relative_to(REPO / 'tasks').parts[:2]
    if (domain, task_group) not in FULL_GROUPS:
        continue
    task = yaml.safe_load(task_path.read_text())
    identity = (domain, task['task_name'])
    TASK_METADATA[identity] = {
        'complexity': variant_complexity(task['task_name']),
        'probes': len(task['mechanism_probes']),
    }
FULL_IDENTITIES = set(TASK_METADATA)
assert len(CORE_IDENTITIES) == 80 and len(FULL_IDENTITIES) == 512


def load_indicators(experiment: str, selected):
    root = RESULTS / experiment
    payloads = {}
    for path in root.glob('*/*/*.json'):
        payload = json.loads(path.read_text())
        evaluation = payload.get('evaluation')
        task_name = (evaluation.get('task_name')
                     if isinstance(evaluation, dict) else path.stem)
        identity = (path.relative_to(root).parts[0], task_name)
        if identity in selected:
            if identity in payloads:
                raise ValueError(f'Duplicate task result: {identity}')
            payloads[identity] = payload

    indicators = []
    for identity in selected:
        evaluation = payloads.get(identity, {}).get('evaluation')
        p_ok = bool(
            isinstance(evaluation, dict)
            and evaluation.get('phenomenal', {}).get('ood_test', {})
            .get('symbolically_equivalent')
        )
        probes = (evaluation.get('mechanism_probes', [])
                  if isinstance(evaluation, dict) else [])
        probe_scores = [
            bool(probe.get('scores', {}).get('ood_test', {})
                 .get('symbolically_equivalent'))
            for probe in probes
        ]
        expected_probes = TASK_METADATA[identity]['probes']
        m_ok = bool(expected_probes > 0 and len(probes) == expected_probes
                    and all(probe_scores))
        indicators.append({
            'identity': identity,
            'complexity': TASK_METADATA[identity]['complexity'],
            'P': p_ok,
            'M': m_ok,
        })
    return indicators


def complexity_series(experiment: str, selected):
    records = load_indicators(experiment, selected)
    output = {'SA(P)': [], 'SA(M)': [], '1-(M|P)': [], 'n': []}
    for complexity in COMPLEXITIES:
        subset = [item for item in records
                  if item['complexity'] == complexity]
        p_count = sum(item['P'] for item in subset)
        m_count = sum(item['M'] for item in subset)
        joint_count = sum(item['P'] and item['M'] for item in subset)
        output['SA(P)'].append(p_count / len(subset))
        output['SA(M)'].append(m_count / len(subset))
        output['1-(M|P)'].append(
            1 - joint_count / p_count if p_count else np.nan)
        output['n'].append(len(subset))
    return {key: np.asarray(value) for key, value in output.items()}


CORE = complexity_series('codex-gpt-5_6-xhigh-coreset-r1', CORE_IDENTITIES)
FULL = complexity_series('codex-gpt-5_6-xhigh-fullset', FULL_IDENTITIES)
assert CORE['n'].sum() == 80 and FULL['n'].sum() == 512
CORE, FULL


In [ ]:
COLORS = {
    'SA(M)': '#d33f6a',
    'SA(P)': '#4a6fe3',
    '1-(M|P)': '#bb65b6',
}
MARKERS = {'SA(M)': 'o', 'SA(P)': 's', '1-(M|P)': '^'}

fig, axes = get_fig(RN=1, CN=2, FW=14, A_ratio=1, HS=0.5, VS=3, LM=5, RM=5, TM=1.5, BM=3, dpi=300, fontsize=14, lw=1, sharex=True, sharey=True)
ax_loader = iter(axes)
legend_handles = {}

for panel_index, (series, title) in enumerate((
    (CORE, 'Core-set (n=80)'),
    (FULL, 'Full-set (n=512)'),
)):
    ax = next(ax_loader)
    ax_right = ax.twinx()

    for metric in ('SA(M)', 'SA(P)'):
        line, = ax.plot(
            COMPLEXITIES, series[metric],
            color=COLORS[metric], marker=MARKERS[metric],
            linewidth=1.5, markersize=5, label=metric,
        )
        legend_handles.setdefault(metric, line)

    conditional = series['1-(M|P)']
    conditional_is_finite = np.isfinite(conditional)
    conditional_line, = ax_right.plot(
        COMPLEXITIES[conditional_is_finite],
        conditional[conditional_is_finite],
        color=COLORS['1-(M|P)'], marker=MARKERS['1-(M|P)'],
        linewidth=1.5, markersize=5, label='1-(M|P)',
    )
    legend_handles.setdefault('1-(M|P)', conditional_line)
    for missing_index in np.flatnonzero(~conditional_is_finite):
        if missing_index > 0 and conditional_is_finite[missing_index - 1]:
            ax_right.plot(
                COMPLEXITIES[missing_index - 1:missing_index + 1],
                [conditional[missing_index - 1], 1.0],
                color=COLORS['1-(M|P)'], linestyle='--',
                linewidth=1.5, marker=None,
            )
            ax_right.annotate(
                '(NA)', xy=(COMPLEXITIES[missing_index], 1.0),
                xytext=(-6, -10), textcoords='offset points',
                color=COLORS['1-(M|P)'], fontsize=10,
                ha='right', va='top',
            )

    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.15, 5.15)
    ax.set_ylim(-0.03, 1.03)
    ax_right.set_ylim(-0.03, 1.03)
    ax.set_xticks(COMPLEXITIES)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax_right.set_yticks(np.linspace(0, 1, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax_right.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.tick_params(axis='both', labelsize=10)
    ax_right.tick_params(axis='y', labelsize=10)
    ax.grid(True, color='#d9d9d9', linewidth=0.7, alpha=0.8)

    if panel_index == 0:
        ax.set_ylabel(r'$\mathrm{SA}(\mathcal{P})$ / $\mathrm{SA}(\mathcal{M})$', fontsize=14)
        ax_right.spines['right'].set_visible(False)
        ax_right.tick_params(axis='y', right=False, labelright=False)
    else:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', left=False, labelleft=False)
        ax_right.set_ylabel(r'$1-(\mathcal{M}\mid\mathcal{P})$', fontsize=14)

fig.supxlabel('Number of Mutations', fontsize=14, y=-0.01)
legend_order = ('SA(M)', 'SA(P)', '1-(M|P)')
legend_labels = (
    r'$\mathrm{SA}(\mathcal{M})$',
    r'$\mathrm{SA}(\mathcal{P})$',
    r'$1-(\mathcal{M}\mid\mathcal{P})$',
)
fig.legend(
    [legend_handles[label] for label in legend_order], legend_labels,
    loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=3,
    frameon=False, fontsize=10, handlelength=2.4, columnspacing=1.6,
)

OUTPUT_STEM = OUTPUT_DIR / 'codex_gpt56_conditional_probability_by_mutations'
for extension in ('png', 'svg', 'pdf'):
    fig.savefig(f'{OUTPUT_STEM}.{extension}', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from matplotlib.lines import Line2D

MODEL_COLORS = {
    'GPT-5.6-sol': '#d33f6a',
    'GLM-5.3-flash': '#4a6fe3',
    'Deepseek-v4-flash-0731': '#bb65b6',
}
GOLD_P_EXPERIMENTS = {
    'GPT-5.6-sol': ('direct_ask_codex-gpt-5_6-sol-medium-reference-p-constants-core-plus-originals',),
    'GLM-5.3-flash': ('reference_phenomenology_direct_ask_glm-5.3-flash',),
    'Deepseek-v4-flash-0731': ('reference_phenomenology_direct_ask_deepseek-v4-flash-0731-low',),
}
STANDARD_CORE_EXPERIMENTS = {
    'GPT-5.6-sol': ('codex-gpt-5_6-xhigh-coreset-r1',),
    'GLM-5.3-flash': (
        'codex-cjl-glm-5.3-flash-high-fullset-r1',
        'codex-cjl-glm-5.3-flash-high-coreset-r2',
        'codex-cjl-glm-5.3-flash-high-coreset-r3',
    ),
    'Deepseek-v4-flash-0731': (
        'codex-cjl-deepseek-v4-flash-0731-low-coreset-r1',
        'codex-cjl-deepseek-v4-flash-0731-low-coreset-r2',
        'codex-cjl-deepseek-v4-flash-0731-low-coreset-r3',
    ),
}
STANDARD_FULL_EXPERIMENTS = {
    'GPT-5.6-sol': ('codex-gpt-5_6-xhigh-fullset',),
    'GLM-5.3-flash': ('codex-cjl-glm-5.3-flash-high-fullset-r1',),
    # No Codex + DeepSeek Full-set run is available.
    'Deepseek-v4-flash-0731': (),
}


def averaged_m_series(experiments, selected):
    if not experiments:
        return np.full(6, np.nan)
    values = [complexity_series(experiment, selected)['SA(M)']
              for experiment in experiments]
    return np.mean(values, axis=0)


panel_data = []
for selected, standard_experiments in (
    (CORE_IDENTITIES, STANDARD_CORE_EXPERIMENTS),
    (FULL_IDENTITIES, STANDARD_FULL_EXPERIMENTS),
):
    panel_data.append({
        'Gold-P Direct Ask': {
            model: averaged_m_series(experiments, selected)
            for model, experiments in GOLD_P_EXPERIMENTS.items()
        },
        'Standard Discovery': {
            model: averaged_m_series(experiments, selected)
            for model, experiments in standard_experiments.items()
        },
    })

fig2, axes2 = get_fig(RN=1, CN=2, FW=14, A_ratio=1, HS=0.5, VS=3, LM=5, RM=5, TM=1.5, BM=4, dpi=300, fontsize=14, lw=1, sharex=True, sharey=True)
ax_loader2 = iter(axes2)
for panel_index, (data, title) in enumerate(zip(
    panel_data, ('Core-set (n=80)', 'Full-set (n=512)')
)):
    ax = next(ax_loader2)
    for method, linestyle in (
        ('Gold-P Direct Ask', '-'),
        ('Standard Discovery', '--'),
    ):
        for model, color in MODEL_COLORS.items():
            values = data[method][model]
            finite = np.isfinite(values)
            if finite.any():
                ax.plot(
                    COMPLEXITIES[finite], values[finite],
                    color=color, linestyle=linestyle, marker='o',
                    linewidth=1.5, markersize=4,
                )
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.15, 5.15)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xticks(COMPLEXITIES)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, color='#d9d9d9', linewidth=0.7, alpha=0.8)
    if panel_index == 0:
        ax.set_ylabel(r'$\mathrm{SA}(\mathcal{M})$', fontsize=14)
    else:
        ax.tick_params(axis='y', left=False, labelleft=False)

fig2.supxlabel('Number of Mutations', fontsize=14, y=0.08)
model_handles = [
    Line2D([0], [0], color=color, linewidth=2, label=model)
    for model, color in MODEL_COLORS.items()
]
method_handles = [
    Line2D([0], [0], color='#777777', linewidth=2, linestyle='-',
           label='Gold-P Direct Ask'),
    Line2D([0], [0], color='#777777', linewidth=2, linestyle='--',
           label='Standard Discovery'),
]
model_legend = fig2.legend(
    handles=model_handles, loc='lower center',
    bbox_to_anchor=(0.5, -0.07), ncol=3, frameon=False,
    fontsize=10, handlelength=2.4, columnspacing=1.5,
)
fig2.add_artist(model_legend)
fig2.legend(
    handles=method_handles, loc='lower center',
    bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False,
    fontsize=10, handlelength=2.4, columnspacing=1.5,
)

OUTPUT_STEM_2 = OUTPUT_DIR / 'gold_p_vs_standard_discovery_sa_m_by_mutations'
for extension in ('png', 'svg', 'pdf'):
    fig2.savefig(f'{OUTPUT_STEM_2}.{extension}', dpi=300, bbox_inches='tight')
plt.show()
